In [2]:
# -*- coding: utf-8 -*-
"""aml adressed concerns with shap layer"""

#!/usr/bin/env python
"""
AML FINAL PIPELINE  -  IBM HI-Small  |  causal features + GNN + XGBoost + validated blend
=====================================================================================

Paste this whole file into ONE Colab cell (GPU runtime, High-RAM) and run it.
Edit the RUN SETTINGS block first. Everything else is automatic.
"""
import argparse
import copy
import json
import os
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, precision_recall_curve
from sklearn.model_selection import KFold, cross_val_predict

try:
    from numba import njit
    HAVE_NUMBA = True
except Exception:                                                      # pragma: no cover
    HAVE_NUMBA = False
from google.colab import drive
drive.mount('/content/drive')
# ========================================================================================
# RUN SETTINGS  (edit these)
# ========================================================================================
CSV_PATH = "/content/drive/MyDrive/ibm aml/HI-Small_Trans.csv"           # raw IBM file (preferred) or your Neo4j export; searched in
                                           # ./  /content  /content/drive/MyDrive ; else an upload prompt
PATTERNS_PATH = "/content/drive/MyDrive/ibm aml/HI-Small_Patterns.txt"   # set None to skip the typology report
OUTPUT_DIR = "/content/aml_outputs" if os.path.isdir("/content") else "aml_outputs"
USE_GNN = True                             # False = tabular only (fast)
STACK_OOF = True                           # True = also run time-forward out-of-fold stacking (+~15 min) (Set True to generate SHAP!)
SMOKE = False                              # True = synthetic data, to test the setup
MOUNT_DRIVE = False                        # True = mount Google Drive first (best for the large raw CSV)
# structural = NO IBM-generator fingerprints (payment format, currency, same-bank, amount equality): the model
#              must learn from behaviour and graph structure. full = everything (highest score on IBM data).
FEATURE_SETS = ["structural", "full"]
GNN_FEATURE_SETS = ["structural"]          # the GNN question is only interesting without the fingerprints
AUDIT_FINGERPRINT_ONLY = True              # also train on the fingerprint columns ALONE to measure the shortcut
SEED = 42

# ---- column names of a non-IBM (Neo4j) export; ignored for the raw IBM file ----
COL_SRC, COL_DST, COL_AMT, COL_TS, COL_LABEL = "source", "target", "amount", "timestamp", "label"
CAT_COLS = ["payment_format", "currency_paid"]                         # one-hot (top 15 by train count)
PATTERN_ID_MODE = "account"                # auto-set to "bank_account" for the raw IBM file

# ---- split / windows ----
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20          # remaining 20% = test
WINDOWS = [3600, 86400, 345600, 604800]    # 1h, 24h, 96h, 168h (laundering attempts last ~1-8 days)
DISTINCT_OWN = [("24h", 86400), ("96h", 345600), ("all", None)]
DISTINCT_CROSS_WINDOW = 345600
MISSING_GAP = 10_000_000                   # "no previous event" gap in seconds (data independent)
BURN_THRESH, BURN_CAP = 0.02, 0.35         # burn-in ends when cold-start share < 2%; at most 35% of train

# ---- GNN ----
NUM_NEIGHBORS = [15, 10, 5, 5]             # use e.g. [10, 8, 4, 3] if slow / out of memory
HIDDEN, LAYERS, DROPOUT = 64, 4, 0.2
EPOCHS, LR = 50, 2e-3
EVAL_EVERY = 10                            # validation-proxy check + best-epoch checkpoint
NEG_RATIO, POS_WEIGHT = 20, 3.0
BATCH_TRAIN, BATCH_EVAL = 2048, 4096
USE_EGO_MARKERS = True
GNN_BURN_IN = False                        # the GNN is data hungry: train it on all train rows
GNN_FOLD_BOUNDS = [0.30, 0.45]             # only used when STACK_OOF = True
NUM_WORKERS = 0

# ---- XGBoost / evaluation ----
SCALE_POS_WEIGHT = 30.0
THRESH_BETA = 1.0                          # threshold maximises F-beta on validation (1.0 -> F1)
N_BOOT = 200
ALERT_BUDGETS = (0.001, 0.005, 0.01)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ========================================================================================
# 1. LOAD
# ========================================================================================
RAW_NAMES = ["Timestamp", "From Bank", "Account", "To Bank", "Account.1", "Amount Received",
             "Receiving Currency", "Amount Paid", "Payment Currency", "Payment Format", "Is Laundering"]


def to_seconds(s):
    t = pd.to_datetime(s)
    return ((t - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)).to_numpy(dtype="int64")


def is_raw_ibm(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        head = fh.readline()
    return "Is Laundering" in head and "Payment Format" in head


def read_table(path):
    """Returns a frame with standard columns; raw IBM files are mapped by position."""
    global PATTERN_ID_MODE
    if is_raw_ibm(path):
        raw = pd.read_csv(path, header=0, names=RAW_NAMES,
                          dtype={"From Bank": str, "Account": str, "To Bank": str, "Account.1": str})
        df = pd.DataFrame({
            COL_SRC: raw["From Bank"] + "_" + raw["Account"],
            COL_DST: raw["To Bank"] + "_" + raw["Account.1"],
            COL_AMT: raw["Amount Paid"], "amount_received": raw["Amount Received"],
            COL_TS: raw["Timestamp"], COL_LABEL: raw["Is Laundering"],
            "payment_format": raw["Payment Format"], "currency_paid": raw["Payment Currency"],
            "currency_received": raw["Receiving Currency"],
            "source_bank": raw["From Bank"], "target_bank": raw["To Bank"]})
        PATTERN_ID_MODE = "bank_account"
        log("  raw IBM file detected: payment format, currencies, banks and amounts are used")
        return df
    log("  WARNING: not the raw IBM file. Payment format / currency / bank fields are missing; in the patterns")
    log("           file laundering is 99.97% ACH, always same-currency, only 2.4% same-bank, so expect a")
    log("           weaker model. Use HI-Small_Trans.csv for the best result.")
    return pd.read_csv(path)


def load_data(path):
    df = read_table(path)
    ts_abs = to_seconds(df[COL_TS])
    order = np.argsort(ts_abs, kind="stable")                          # chronological, ties keep file order
    df = df.iloc[order].reset_index(drop=True)
    ts_abs = ts_abs[order]
    n = len(df)
    src_raw = df[COL_SRC].astype(str).to_numpy()
    dst_raw = df[COL_DST].astype(str).to_numpy()
    codes, uniques = pd.factorize(np.concatenate([src_raw, dst_raw]))
    src, dst = codes[:n].astype(np.int64), codes[n:].astype(np.int64)
    y = df[COL_LABEL].fillna(0).astype(int).to_numpy().astype(np.int64)
    amt = df[COL_AMT].fillna(0.0).clip(lower=0).to_numpy(dtype="float64")
    pos = np.where(y == 1)[0]
    return dict(df=df, n=n, N=len(uniques), src=src, dst=dst, y=y, amt=amt, ts=ts_abs - ts_abs.min(),
                ts_abs=ts_abs, src_raw_pos=src_raw[pos], dst_raw_pos=dst_raw[pos], pos_idx=pos)


# ========================================================================================
# 2. CAUSAL FEATURES (every value uses only events strictly before the transaction's time)
# ========================================================================================
def build_stream(acct, ts, amt, big):
    key = acct.astype(np.int64) * big + ts
    order = np.argsort(key, kind="stable")
    cs = np.concatenate([[0.0], np.cumsum(amt[order], dtype=np.float64)])
    return key[order], cs, amt[order]


def hist_query(stream, q_acct, q_ts, big, window=None):
    key_s, cs, _ = stream
    base = q_acct.astype(np.int64) * big
    hi = np.searchsorted(key_s, base + q_ts, side="left")
    lo_key = base if window is None else base + np.maximum(q_ts - window, 0)
    lo = np.searchsorted(key_s, lo_key, side="left")
    return (hi - lo).astype(np.float32), (cs[hi] - cs[lo]).astype(np.float32)


def last_event(stream, q_acct, q_ts, big):
    key_s, _, amt_s = stream
    base = q_acct.astype(np.int64) * big
    hi = np.searchsorted(key_s, base + q_ts, side="left")
    lo = np.searchsorted(key_s, base, side="left")
    has = hi > lo
    idx = np.maximum(hi - 1, 0)
    gap = np.where(has, q_ts - (key_s[idx] - base), MISSING_GAP).astype(np.float32)
    return gap, np.where(has, amt_s[idx], 0.0), has


def first_seen(codes, ts, N):
    u, idx = np.unique(codes, return_index=True)                      # ts sorted -> first index = earliest
    out = np.full(N, MISSING_GAP, dtype=np.int64)
    out[u] = ts[idx]
    return out


# ---- exact distinct-counterparty counts in [t - window, t) via a two-pointer sweep ----
def _distinct_sweep(seg_start, seg_end, ev_ts, ev_partner, q_acct, q_ts, nq, window, cnt, out):
    cur = -1
    lo = 0
    hi = 0
    distinct = 0
    for k in range(nq):
        a = q_acct[k]
        if a != cur:
            for j in range(lo, hi):                                    # empty the previous account's window
                cnt[ev_partner[j]] -= 1
            distinct = 0
            lo = seg_start[a]
            hi = seg_start[a]
            cur = a
        t = q_ts[k]
        e = seg_end[a]
        while hi < e and ev_ts[hi] < t:
            c = ev_partner[hi]
            if cnt[c] == 0:
                distinct += 1
            cnt[c] += 1
            hi += 1
        lim = t - window
        while lo < hi and ev_ts[lo] < lim:
            c = ev_partner[lo]
            cnt[c] -= 1
            if cnt[c] == 0:
                distinct -= 1
            lo += 1
        out[k] = distinct


_SWEEP_NB = njit(cache=False)(_distinct_sweep) if HAVE_NUMBA else None
_NUMBA_OK = {"ok": HAVE_NUMBA}


def stream_struct(acct, partner, ts, N):
    order = np.argsort(acct, kind="stable")                            # stable keeps time order per account
    a_s = acct[order]
    ids = np.arange(N)
    return (np.searchsorted(a_s, ids, "left").astype(np.int64), np.searchsorted(a_s, ids, "right").astype(np.int64),
            ts[order].astype(np.int64), partner[order].astype(np.int64))


def query_struct(q_acct, q_ts):
    order = np.argsort(q_acct, kind="stable")
    return order, q_acct[order].astype(np.int64), q_ts[order].astype(np.int64)


def distinct_counts(stream, qs, N, window):
    seg_start, seg_end, ev_ts, ev_partner = stream
    order, qa, qt = qs
    win = (1 << 60) if window is None else int(window)
    nq = len(qa)
    done = False
    if _NUMBA_OK["ok"]:
        try:
            out_s = np.zeros(nq, dtype=np.int64)
            _SWEEP_NB(seg_start, seg_end, ev_ts, ev_partner, qa, qt, nq, win, np.zeros(N, dtype=np.int64), out_s)
            done = True
        except Exception as e:                                         # numba unavailable for this interpreter
            log(f"  numba sweep failed ({type(e).__name__}); using the slower pure-Python path")
            _NUMBA_OK["ok"] = False
    if not done:
        out_l = [0] * nq
        _distinct_sweep(seg_start.tolist(), seg_end.tolist(), ev_ts.tolist(), ev_partner.tolist(),
                        qa.tolist(), qt.tolist(), nq, win, [0] * N, out_l)
        out_s = np.asarray(out_l, dtype=np.int64)
    out = np.empty(nq, dtype=np.float32)
    out[order] = out_s
    return out


def causal_features(d):
    src, dst, ts, amt, N, n = d["src"], d["dst"], d["ts"], d["amt"], d["N"], d["n"]
    big = int(ts.max()) + max(WINDOWS) + 2
    out_stream = build_stream(src, ts, amt, big)                       # money sent, keyed by sender
    in_stream = build_stream(dst, ts, amt, big)                        # money received, keyed by receiver
    cf, cr = src * N + dst, dst * N + src
    uniq = np.unique(np.concatenate([cf, cr]))
    pid_f, pid_r = np.searchsorted(uniq, cf), np.searchsorted(uniq, cr)
    pair_stream = build_stream(pid_f, ts, amt, big)

    f, logged = {}, set()

    def add(name, arr, log1p=False):
        f[name] = np.asarray(arr, dtype=np.float32)
        if log1p:
            logged.add(name)

    for pre, acct in (("s", src), ("r", dst)):                         # s = sender, r = receiver
        for tag, stream in (("out", out_stream), ("in", in_stream)):
            c, s = hist_query(stream, acct, ts, big)
            add(f"{pre}_{tag}_cnt", c, True)
            add(f"{pre}_{tag}_sum", s, True)
            for w in WINDOWS:
                c, s = hist_query(stream, acct, ts, big, w)
                add(f"{pre}_{tag}_cnt_{w // 3600}h", c, True)
                add(f"{pre}_{tag}_sum_{w // 3600}h", s, True)
            gap, last_amt, has = last_event(stream, acct, ts, big)
            add(f"{pre}_gap_{tag}", gap, True)
            if pre == "s" and tag == "in":                             # was this money just forwarded?
                add("s_fwd_ratio", np.where(has, np.clip(amt / (last_amt + 1.0), 0, 10), 0.0))

    eps = 1.0
    add("s_imb", np.abs(f["s_in_sum"] - f["s_out_sum"]) / (f["s_in_sum"] + f["s_out_sum"] + eps))
    add("r_imb", np.abs(f["r_in_sum"] - f["r_out_sum"]) / (f["r_in_sum"] + f["r_out_sum"] + eps))
    for w in WINDOWS[1:]:
        h = f"{w // 3600}h"
        for p in ("s", "r"):
            i_, o_ = f[f"{p}_in_sum_{h}"], f[f"{p}_out_sum_{h}"]
            add(f"{p}_imb_{h}", np.abs(i_ - o_) / (i_ + o_ + eps))
            add(f"{p}_pass_{h}", np.clip(o_ / (i_ + eps), 0, 10))
    add("amt_vs_s_avg", np.clip(amt / (f["s_out_sum"] / np.maximum(f["s_out_cnt"], 1) + eps), 0, 50), True)
    add("amt_vs_r_avg", np.clip(amt / (f["r_in_sum"] / np.maximum(f["r_in_cnt"], 1) + eps), 0, 50), True)

    c, _ = hist_query(pair_stream, pid_f, ts, big)
    add("pair_fwd_cnt", c, True)
    c, _ = hist_query(pair_stream, pid_f, ts, big, 345600)
    add("pair_fwd_cnt_96h", c, True)
    c, s = hist_query(pair_stream, pid_r, ts, big)                     # money that already flowed r -> s
    add("pair_rev_cnt", c, True)
    add("pair_rev_sum", s, True)
    c, _ = hist_query(pair_stream, pid_r, ts, big, 345600)
    add("pair_rev_cnt_96h", c, True)

    # breadth: how many DIFFERENT accounts (fan-out of the sender, fan-in of the receiver) in the window
    ev_out, ev_in = stream_struct(src, dst, ts, N), stream_struct(dst, src, ts, N)
    q_src, q_dst = query_struct(src, ts), query_struct(dst, ts)
    for tag, w in DISTINCT_OWN:
        add(f"s_out_nuniq_{tag}", distinct_counts(ev_out, q_src, N, w), True)
        add(f"r_in_nuniq_{tag}", distinct_counts(ev_in, q_dst, N, w), True)
    add("r_out_nuniq_96h", distinct_counts(ev_out, q_dst, N, DISTINCT_CROSS_WINDOW), True)
    add("s_in_nuniq_96h", distinct_counts(ev_in, q_src, N, DISTINCT_CROSS_WINDOW), True)

    first = np.minimum(first_seen(src, ts, N), first_seen(dst, ts, N))
    add("s_age", ts - first[src], True)                                # causal: first sighting <= now
    add("r_age", ts - first[dst], True)
    extras = {"s_new": ((f["s_out_cnt"] + f["s_in_cnt"]) == 0),        # diagnostics / burn-in only
              "r_new": ((f["r_out_cnt"] + f["r_in_cnt"]) == 0)}

    hour = (d["ts_abs"] % 86400) / 86400.0 * 2 * np.pi
    add("log_amount", amt, True)
    add("sin_hour", np.sin(hour))
    add("cos_hour", np.cos(hour))
    add("self_loop", src == dst)

    df = d["df"]
    if "amount_received" in df.columns:
        rec = df["amount_received"].fillna(0.0).clip(lower=0).to_numpy(dtype="float64")
        add("amt_equal", np.isclose(rec, amt, rtol=1e-6, atol=1e-6))
        add("amt_logratio", np.abs(np.log((rec + 1.0) / (amt + 1.0))))
    for col in CAT_COLS:
        if col in df.columns:
            codes, names_ = pd.factorize(df[col].astype(str))
            counts = np.bincount(codes[:int(TRAIN_FRAC * n)], minlength=len(names_))
            keep = sorted(range(len(names_)), key=lambda k: (-counts[k], str(names_[k])))[:15]
            for k in keep:
                add(f"{col}={names_[k]}", codes == k)
    if "source_bank" in df.columns and "target_bank" in df.columns:
        add("same_bank", (df["source_bank"].astype(str) == df["target_bank"].astype(str)).to_numpy())
    if "currency_paid" in df.columns and "currency_received" in df.columns:
        add("same_currency", (df["currency_paid"].astype(str) == df["currency_received"].astype(str)).to_numpy())

    names = list(f.keys())
    X = np.empty((n, len(names)), dtype=np.float32)
    for j, k in enumerate(names):
        v = f[k]
        X[:, j] = np.log1p(np.maximum(v, 0)) if k in logged else v
    return X, names, extras


def choose_burn_in(cold, tr_end, n):
    chunk = max(n // 100, 1)
    rates = [float(cold[i:i + chunk].mean()) for i in range(0, tr_end, chunk)]
    bad = [i for i, r in enumerate(rates) if r >= BURN_THRESH]
    burn_end = min(((bad[-1] + 1) * chunk) if bad else 0, int(BURN_CAP * tr_end))
    return burn_end, rates


# ========================================================================================
# 3. METRICS
# ========================================================================================
def best_threshold(y, p, beta):
    pre, rec, thr = precision_recall_curve(y, p)
    b2 = beta ** 2
    fb = (1 + b2) * pre[:-1] * rec[:-1] / (b2 * pre[:-1] + rec[:-1] + 1e-12)
    return float(thr[int(np.argmax(fb))])


def prf(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum())
    p = tp / max(int(pred.sum()), 1)
    r = tp / max(int(y.sum()), 1)
    return p, r, 2 * p * r / (p + r + 1e-12)


class Ranked:
    """Pre-sorted scores so a bootstrap PR-AUC is O(n) cumsums instead of a re-sort."""

    def __init__(self, y, p):
        self.order = np.argsort(-p, kind="stable")
        self.y = y[self.order].astype(np.float64)

    def ap(self, w):
        wo = w[self.order]
        pos = wo * self.y
        s = pos.sum()
        if s <= 0:
            return float("nan")
        return float((pos * np.cumsum(pos) / np.maximum(np.cumsum(wo), 1e-12)).sum() / s)


def boot_weights(n, rng, block=0):
    if block:
        nb = max(n // block, 1)
        starts = rng.integers(0, max(n - block, 1), nb)
        idx = (starts[:, None] + np.arange(block)[None, :]).ravel()
    else:
        idx = rng.integers(0, n, n)
    return np.bincount(idx, minlength=n).astype(np.float64)


def bootstrap_ap(y, p, n_boot, seed=0, block=0):
    rng, rk = np.random.default_rng(seed), Ranked(y, p)
    aps = [rk.ap(boot_weights(len(y), rng, block)) for _ in range(n_boot)]
    aps = [a for a in aps if a == a]
    return tuple(np.percentile(aps, [2.5, 97.5])) if aps else (float("nan"), float("nan"))


def paired_bootstrap_ap(y, p_a, p_b, n_boot, seed=0, block=0):
    rng, ra, rb = np.random.default_rng(seed), Ranked(y, p_a), Ranked(y, p_b)
    diffs = []
    for _ in range(n_boot):
        w = boot_weights(len(y), rng, block)
        a, b = ra.ap(w), rb.ap(w)
        if a == a and b == b:
            diffs.append(a - b)
    diffs = np.asarray(diffs)
    return float(diffs.mean()), tuple(np.percentile(diffs, [2.5, 97.5])), float((diffs > 0).mean())


def make_run(name, short, p_va, p_te, yva, yte, model=None, extra=None):
    thr = best_threshold(yva, p_va, THRESH_BETA)
    pr, rc, f1 = prf(yte, (p_te >= thr).astype(int))
    r = dict(name=name, short=short, ap_val=float(average_precision_score(yva, p_va)),
             ap_test=float(average_precision_score(yte, p_te)), thr=thr, precision=pr, recall=rc, f1=f1,
             p_va=p_va, p_te=p_te, model=model)
    if extra:
        r.update(extra)
    return r


def fit_xgb(name, short, Xtr, ytr, Xva, yva, Xte, yte):
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        device = "cpu"
    log(f"XGBoost [{name}]  train={len(ytr):,} (pos {int(ytr.sum()):,})  device={device}")
    clf = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=SCALE_POS_WEIGHT, tree_method="hist", device=device,
        eval_metric="aucpr", early_stopping_rounds=100, random_state=SEED)
    clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=200)
    return make_run(name, short, clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1], yva, yte,
                    model=clf, extra=dict(best_iter=int(clf.best_iteration)))


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def blend_run(run_tab, g_va, g_te, yva, yte):
    """Logistic blend of tabular logit + GNN logit. Weights are fitted on VALIDATION; the validation score
    used for model selection and the threshold is the 5-fold out-of-fold score, so it is not self-graded."""
    Zv = np.column_stack([logit(run_tab["p_va"]), g_va])
    Zt = np.column_stack([logit(run_tab["p_te"]), g_te])
    mu, sd = Zv.mean(0), Zv.std(0) + 1e-9
    Zv, Zt = (Zv - mu) / sd, (Zt - mu) / sd
    lr = LogisticRegression(C=1.0, max_iter=1000)
    oof = cross_val_predict(lr, Zv, yva, cv=KFold(5, shuffle=True, random_state=SEED), method="decision_function")
    lr.fit(Zv, yva)
    w = lr.coef_[0]
    log(f"  blend weights (standardised logits): tabular {w[0]:+.2f} | GNN {w[1]:+.2f}")
    return make_run("blend (tabular + GNN, fitted on val)", "blend", oof, lr.decision_function(Zt), yva, yte,
                    extra=dict(weights=[float(w[0]), float(w[1])]))


# ========================================================================================
# 4. TYPOLOGIES (HI-Small_Patterns.txt)
# ========================================================================================
def load_typologies(path, d):
    rows, cur = [], None
    with open(path, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):
                cur = line.split(" - ", 1)[1].split(":")[0].strip()
            elif line.startswith("END LAUNDERING ATTEMPT"):
                cur = None
            elif cur and line:
                p = line.split(",")
                s_id = p[2] if PATTERN_ID_MODE == "account" else f"{p[1]}_{p[2]}"
                d_id = p[4] if PATTERN_ID_MODE == "account" else f"{p[3]}_{p[4]}"
                rows.append((cur, p[0], s_id, d_id, float(p[5]), float(p[7])))
    pat = pd.DataFrame(rows, columns=["typology", "ts", "src", "dst", "amt_recv", "amt_paid"])
    pat["ts"] = to_seconds(pd.to_datetime(pat["ts"], format="%Y/%m/%d %H:%M"))
    left = pd.DataFrame({"row": d["pos_idx"], "ts": d["ts_abs"][d["pos_idx"]], "src": d["src_raw_pos"],
                         "dst": d["dst_raw_pos"], "amt": d["amt"][d["pos_idx"]]})
    m = left.merge(pat, on=["ts", "src", "dst"])
    ok = np.isclose(m.amt, m.amt_recv, rtol=1e-3, atol=0.01) | np.isclose(m.amt, m.amt_paid, rtol=1e-3, atol=0.01)
    m = m[ok].drop_duplicates("row")
    typ = np.full(d["n"], "", dtype=object)
    typ[m["row"].to_numpy()] = m["typology"].to_numpy()
    log(f"  typology match: {len(m):,} of {len(d['pos_idx']):,} labelled laundering rows "
        f"({len(m) / max(len(d['pos_idx']), 1):.0%}); the patterns file itself lists {len(pat):,} transactions")
    if len(m) < 0.5 * len(pat):
        log("  WARNING: few pattern rows matched - check PATTERN_ID_MODE and that the same dataset variant is used")
    return typ


# ========================================================================================
# 5. GNN STAGE
# ========================================================================================
def setup_gnn_deps():
    import subprocess
    import sys
    try:
        import torch
    except ImportError:
        log("PyTorch is not installed - GNN stage unavailable")
        return False
    try:
        import torch_geometric  # noqa: F401
        import pyg_lib  # noqa: F401
        return True
    except ImportError:
        pass
    ver = torch.__version__.split("+")[0]
    cu = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
    url = f"https://data.pyg.org/whl/torch-{ver}+{cu}.html"
    log(f"Installing torch_geometric + pyg-lib for torch {ver}+{cu} ...")
    for cmd in ([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"],
                [sys.executable, "-m", "pip", "install", "-q", "pyg-lib", "-f", url]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            log("  install failed: " + (r.stderr.strip().splitlines() or ["unknown error"])[-1])
    try:
        import torch_geometric  # noqa: F401
        import pyg_lib  # noqa: F401
        return True
    except ImportError:
        log(f"  pyg-lib is unavailable for torch {ver}+{cu} (see {url}); GNN stage will be skipped")
        return False


def run_gnn_stage(X, d, tr_end, va_end, fit_start):
    import torch
    import torch.nn as nn
    import torch.nn.functional as Fn
    from torch_geometric.data import Data
    from torch_geometric.loader import LinkNeighborLoader
    from torch_geometric.nn import TransformerConv

    n, N, y = d["n"], d["N"], d["y"]
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    log(f"GNN stage on {dev}")
    if not np.isfinite(X).all():
        raise ValueError("non-finite values in the feature matrix")

    X_t = torch.from_numpy(X)
    src_t, dst_t = torch.from_numpy(d["src"]), torch.from_numpy(d["dst"])
    ts_t, y_t = torch.from_numpy(d["ts"]), torch.from_numpy(y)
    edge_attr = torch.cat([torch.cat([X_t, torch.zeros(n, 1)], 1),                 # forward edges, flag 0
                           torch.cat([X_t, torch.ones(n, 1)], 1)], 0)              # reverse edges, flag 1
    edge_index = torch.cat([torch.stack([src_t, dst_t]), torch.stack([dst_t, src_t])], 1)
    data = Data(x=torch.ones(N, 1), edge_index=edge_index, edge_attr=edge_attr,
                time=torch.cat([ts_t, ts_t]), num_nodes=N)

    class EdgeGNN(nn.Module):
        def __init__(self, edge_dim, own_dim):
            super().__init__()
            self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
            for i in range(LAYERS):
                self.convs.append(TransformerConv(3 if i == 0 else HIDDEN, HIDDEN, edge_dim=edge_dim))
                self.bns.append(nn.BatchNorm1d(HIDDEN))
            self.own = nn.Sequential(nn.Linear(own_dim, HIDDEN), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(3 * HIDDEN, HIDDEN), nn.ReLU(),
                                      nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 1))

        def forward(self, x, ei, ea, seed, own):
            h = x
            for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
                h2 = Fn.dropout(Fn.relu(bn(conv(h, ei, ea))), p=DROPOUT, training=self.training)
                h = h2 + h if i > 0 else h2
            z = torch.cat([h[seed[0]], h[seed[1]], self.own(own)], dim=-1)
            return self.head(z).squeeze(-1)

    def make_loader(rows, bs):
        rows_t = torch.as_tensor(rows, dtype=torch.long)
        loader = LinkNeighborLoader(
            data, num_neighbors=NUM_NEIGHBORS, time_attr="time",
            edge_label_index=torch.stack([src_t[rows_t], dst_t[rows_t]]),
            edge_label=y_t[rows_t].float(),
            edge_label_time=ts_t[rows_t] - 1,                                      # strictly earlier edges only
            batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
        return loader, rows_t

    checked = {"done": False}

    def run_batch(model, batch, rows_t):
        batch = batch.to(dev)
        g = rows_t[batch.input_id.cpu()]
        x = torch.zeros(batch.x.size(0), 3, device=dev)
        x[:, 0] = 1
        if USE_EGO_MARKERS:
            x[batch.edge_label_index[0], 1] = 1
            x[batch.edge_label_index[1], 2] = 1
        logits = model(x, batch.edge_index, batch.edge_attr, batch.edge_label_index, X_t[g].to(dev))
        if not checked["done"]:
            assert torch.equal(batch.edge_label.long().cpu(), y_t[g]), "seed/label misalignment"
            bt = getattr(batch, "time", None)
            if bt is not None:
                assert int(bt.max()) <= int(ts_t[g].max()) - 1, "sampler returned future edges"
            checked["done"] = True
            log("  sanity checks passed (seed alignment, temporal sampling)")
        return logits, batch

    @torch.no_grad()
    def predict(model, rows):
        model.eval()
        out = np.zeros(len(rows), dtype=np.float32)
        loader, rows_t = make_loader(rows, BATCH_EVAL)
        for batch in loader:
            logits, b = run_batch(model, batch, rows_t)
            out[b.input_id.cpu().numpy()] = logits.float().cpu().numpy()
        return out

    def train(fit_rows, tag, monitor_rows=None):
        pos, neg = fit_rows[y[fit_rows] == 1], fit_rows[y[fit_rows] == 0]
        if len(pos) < 20:
            raise RuntimeError(f"{tag}: only {len(pos)} positives to train on")
        log(f"  [{tag}] fit rows {len(fit_rows):,} | positives {len(pos):,}")
        model = EdgeGNN(X.shape[1] + 1, X.shape[1]).to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=dev))
        best = {"ap": -1.0, "ep": 0, "state": None}
        for ep in range(1, EPOCHS + 1):
            model.train()
            k = min(len(neg), NEG_RATIO * len(pos))
            seeds = np.concatenate([pos, rng.choice(neg, k, replace=False)])  # fresh negatives every epoch
            rng.shuffle(seeds)
            loader, rows_t = make_loader(seeds, BATCH_TRAIN)
            tot, nb = 0.0, 0
            for batch in loader:
                opt.zero_grad()
                logits, b = run_batch(model, batch, rows_t)
                loss = crit(logits, b.edge_label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot, nb = tot + loss.item(), nb + 1
            sched.step()
            if ep % 5 == 0 or ep == 1:
                log(f"  [{tag}] epoch {ep:02d}/{EPOCHS}  loss {tot / max(nb, 1):.4f}")
            if monitor_rows is not None and (ep % EVAL_EVERY == 0 or ep == EPOCHS):
                ap = average_precision_score(y[monitor_rows], predict(model, monitor_rows))
                log(f"  [{tag}] epoch {ep:02d} validation-proxy PR-AUC {ap:.4f}")
                if ap > best["ap"]:
                    best.update(ap=ap, ep=ep, state=copy.deepcopy(model.state_dict()))
        if best["state"] is not None:
            model.load_state_dict(best["state"])
            log(f"  [{tag}] restored best epoch {best['ep']} (proxy PR-AUC {best['ap']:.4f})")
        return model

    out = {"final": np.full(n, np.nan, dtype=np.float32), "oof": None, "s0": None}
    if STACK_OOF:                                                                  # time-forward folds
        out["oof"] = np.full(n, np.nan, dtype=np.float32)
        bounds = [int(b * n) for b in GNN_FOLD_BOUNDS] + [tr_end]
        out["s0"] = bounds[0]
        for k in range(len(bounds) - 1):
            model = train(np.arange(0, bounds[k]), f"fold{k + 1}")
            rows = np.arange(bounds[k], bounds[k + 1])
            out["oof"][rows] = predict(model, rows)
            if y[rows].sum():
                log(f"  [fold{k + 1}] standalone OOF PR-AUC {average_precision_score(y[rows], out['oof'][rows]):.4f}")
            del model

    val_rows = np.arange(tr_end, va_end)                                           # proxy for epoch selection
    vpos = val_rows[y[val_rows] == 1]
    vneg = rng.choice(val_rows[y[val_rows] == 0], min(100_000, int((y[val_rows] == 0).sum())), replace=False)
    monitor = np.sort(np.concatenate([vpos, vneg]))
    model = train(np.arange(fit_start, tr_end), "final", monitor_rows=monitor)
    rows = np.arange(tr_end, n)
    out["final"][rows] = predict(model, rows)
    log(f"  [final] standalone GNN PR-AUC  val {average_precision_score(y[tr_end:va_end], out['final'][tr_end:va_end]):.4f}"
        f"  test {average_precision_score(y[va_end:], out['final'][va_end:]):.4f}")
    return out


# ========================================================================================
# 6. REPORTING
# ========================================================================================
def print_importance(run, names, k=20):
    imp = run["model"].get_booster().get_score(importance_type="gain")
    print(f"\nTop features by GAIN ({run['name']}):")
    for key, v in sorted(imp.items(), key=lambda kv: -kv[1])[:k]:
        idx = int(key[1:]) if key.startswith("f") and key[1:].isdigit() else None
        print(f"  {names[idx] if idx is not None and idx < len(names) else key:<28} {v:.1f}")


def diagnostics(res, y_va, y_te):
    p_te, p_va, thr = res["p_te"], res["p_va"], res["thr"]
    pred = (p_te >= thr).astype(int)
    print("\n" + "=" * 80 + f"\nHEADLINE (best validation PR-AUC): {res['name']}\n" + "=" * 80)
    print(classification_report(y_te, pred, target_names=["Licit", "Illicit"], digits=4, zero_division=0))
    lo, hi = bootstrap_ap(y_te, p_te, N_BOOT)
    blo, bhi = bootstrap_ap(y_te, p_te, N_BOOT, block=max(len(y_te) // 100, 1))
    base = y_te.mean()
    print(f"Test PR-AUC {res['ap_test']:.4f}  (95% CI iid {lo:.4f}-{hi:.4f}; time-block {blo:.4f}-{bhi:.4f}; "
          f"base rate {base:.5f}; lift {res['ap_test'] / base:.0f}x)")
    print("  (the time-block CI is the honest one: laundering edges come in clusters)")
    print(f"Val  PR-AUC {res['ap_val']:.4f}   threshold (picked on val, beta={THRESH_BETA}) = {thr:.5f}")
    print(f"Alerts on test: {int(pred.sum()):,} of {len(pred):,} ({pred.mean() * 100:.3f}%)")
    order = np.argsort(-p_te)
    print("\nAlert-budget view (what a reviewer team would see):")
    for fr in ALERT_BUDGETS:
        k = max(int(fr * len(y_te)), 1)
        tp = int(y_te[order[:k]].sum())
        print(f"  top {fr * 100:.1f}% ({k:,} alerts): precision {tp / k:.3f}  recall {tp / max(y_te.sum(), 1):.3f}")
    print("\nDrift check - PR-AUC over time slices:")
    for nm, y_, p_, parts in (("val", y_va, p_va, 2), ("test", y_te, p_te, 4)):
        out = []
        for chunk in np.array_split(np.arange(len(y_)), parts):
            ap_ = f"{average_precision_score(y_[chunk], p_[chunk]):.3f}" if y_[chunk].sum() else "n/a"
            out.append(f"{ap_} (pos {int(y_[chunk].sum())}, {y_[chunk].mean() * 100:.2f}%)")
        print(f"  {nm}: " + "   ".join(out))


def typology_table(runs, y_te, typ_te):
    pos = y_te == 1
    names = sorted(set(typ_te[pos]))
    k = max(int(0.01 * len(y_te)), 1)
    tops = {r["short"]: np.isin(np.arange(len(y_te)), np.argsort(-r["p_te"])[:k]) for r in runs}
    preds = {r["short"]: r["p_te"] >= r["thr"] for r in runs}
    head = "".join(f"{r['short'][:9]:>11}" for r in runs)
    print("\nRecall by laundering typology on TEST   [recall at validation threshold | recall in top 1% alerts]")
    print(f"  {'typology':<22}{'n':>5}{head}")
    for t in names:
        m = pos & (typ_te == t)
        cells = "".join(f"  {preds[r['short']][m].mean():.2f}|{tops[r['short']][m].mean():.2f}" for r in runs)
        print(f"  {(t or '(not in patterns file)'):<22}{int(m.sum()):>5}{cells}")


# ========================================================================================
# 7. SYNTHETIC DATA (for SMOKE) - written in the raw IBM format, plus a patterns file
# ========================================================================================
def make_synthetic(path, patterns_path, n_bg=250_000, n_struct=90, seed=0):
    rng = np.random.default_rng(seed)
    n_acc = 6000
    w = 1.0 / np.arange(1, n_acc + 1) ** 0.8
    w /= w.sum()
    t0 = pd.Timestamp("2022-09-01")
    fmts = np.array(["Reinvestment", "Cheque", "Credit Card", "ACH", "Cash", "Wire", "Bitcoin"])
    fp = np.array([.15, .2, .2, .25, .1, .08, .02])
    cur = np.array(["US Dollar", "Euro", "Yen", "Bitcoin"])
    s, r = rng.choice(n_acc, n_bg, p=w), rng.choice(n_acc, n_bg, p=w)
    minutes = np.sort(rng.integers(0, 10 * 1440, n_bg))
    amt = rng.lognormal(7, 1.5, n_bg)
    fm = rng.choice(fmts, n_bg, p=fp)
    cross = rng.random(n_bg) < 0.12
    bank = lambda a: f"{(a % 40):03d}"
    acct = lambda a: f"8{a:07X}"
    rows = []
    for i in range(n_bg):
        c1 = cur[rng.integers(0, 3)]
        c2 = cur[rng.integers(0, 3)] if cross[i] else c1
        rows.append((t0 + pd.Timedelta(minutes=int(minutes[i])), bank(s[i]), acct(s[i]), bank(r[i]), acct(r[i]),
                     amt[i] * (1.1 if cross[i] else 1.0), c2, amt[i], c1, fm[i], 0))
    pats, nxt = [], n_acc
    for k in range(n_struct):                                       # scatter-gather via fresh mules
        hub, sink = int(rng.integers(0, n_acc)), int(rng.integers(0, n_acc))
        start, tot = int(rng.integers(0, 9 * 1440)), float(rng.lognormal(10, 1))
        edges = []
        for m in range(6):
            mule, a = nxt, tot / 6 * rng.uniform(0.95, 1.05)
            nxt += 1
            t1 = start + int(rng.integers(0, 10))
            edges.append((t1, hub, mule, a))
            edges.append((t1 + int(rng.integers(5, 120)), mule, sink, a * 0.98))
        for (tm, a1, a2, a) in edges:
            row = (t0 + pd.Timedelta(minutes=tm), bank(a1), acct(a1), bank(a2), acct(a2), a, "US Dollar", a,
                   "US Dollar", "ACH", 1)
            rows.append(row)
            pats.append((k, row))
    df = pd.DataFrame(rows, columns=RAW_NAMES).sample(frac=1.0, random_state=seed)
    df["Timestamp"] = df["Timestamp"].dt.strftime("%Y/%m/%d %H:%M")
    df.to_csv(path, index=False)                                    # pandas writes the duplicate "Account" header
    with open(patterns_path, "w") as fh:
        for k in sorted({a for a, _ in pats}):
            fh.write("BEGIN LAUNDERING ATTEMPT - SCATTER-GATHER:  Max 2 hops\n")
            for a, row in pats:
                if a == k:
                    fh.write(",".join([row[0].strftime("%Y/%m/%d %H:%M"), row[1], row[2], row[3], row[4],
                                       f"{row[5]:.2f}", row[6], f"{row[7]:.2f}", row[8], row[9], "1"]) + "\n")
            fh.write("END LAUNDERING ATTEMPT - SCATTER-GATHER\n")
    log(f"Synthetic raw-format data: {len(df):,} rows, {int(df['Is Laundering'].sum()):,} laundering")

# ========================================================================================
# 8. MAIN & EXPLAINABILITY
# ========================================================================================
def find_or_upload(path, required=True):
    if path is None:
        return None
    for cand in (path, os.path.join("/content", os.path.basename(path)),
                 os.path.join("/content/drive/MyDrive", os.path.basename(path))):
        if os.path.exists(cand):
            return cand
    if not required:
        log(f"'{path}' not found - skipping (optional)")
        return None
    try:
        from google.colab import files
        log(f"'{path}' not found - choose the file to upload ...")
        up = files.upload()
        if not up:
            raise FileNotFoundError(f"no file uploaded for {path}")
        return next(iter(up))
    except ImportError:
        raise FileNotFoundError(f"file not found: {path}")

def is_fingerprint(nm):
    """Columns that identify laundering in the IBM generator (payment format, currency, amount equality,
    same-bank) but would not be this clean in real transaction streams."""
    return nm.startswith(("payment_format=", "currency_paid=")) or nm in (
        "same_currency", "amt_equal", "amt_logratio", "same_bank")

def generate_explainability(runs, X, gnn_out, names, yte, typ, va_end, output_dir, tag):
    try:
        import shap
        import matplotlib.pyplot as plt
    except ImportError:
        log("SHAP or matplotlib not installed. Skipping explainability layer.")
        return

    log(f"[{tag}] Generating SHAP Explainability Layer...")

    # 1. Isolate the Stacked OOF Model
    try:
        stack_run = next(r for r in runs if r["short"] == "stack")
    except StopIteration:
        log("  Stacked model not found. Run with --stack to generate SHAP plots.")
        return

    model = stack_run["model"]

    # 2. Reconstruct the precise test matrix (Tabular + GNN Score)
    X_test_tab = X[va_end:]
    gnn_test = gnn_out["final"][va_end:]
    X_test_stacked = np.hstack([X_test_tab, gnn_test[:, None]])
    feature_names = names + ["gnn_score"]

    # 3. Initialize TreeExplainer (Hardware Accelerated)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X_test_stacked)

    # 4. Global Summary Plot (Macro View)
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_stacked, feature_names=feature_names, show=False)
    plt.title(f"Global Feature Impact (SHAP Summary) - {tag}")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"shap_summary_global_{tag}.png"), dpi=300, bbox_inches='tight')
    plt.close()
    log(f"  -> Saved Global SHAP plot: {output_dir}/shap_summary_global_{tag}.png")

    # 5. Local Waterfall Plot (Micro View: Typological Rescue)
    if typ is not None:
        # Find a True Positive "GATHER-SCATTER" transaction
        pos_idx = np.where(yte == 1)[0]
        preds = stack_run["p_te"] >= stack_run["thr"]

        caught_mask = preds[pos_idx] == True
        gs_mask = typ[pos_idx] == "GATHER-SCATTER"
        valid_targets = pos_idx[caught_mask & gs_mask]

        if len(valid_targets) > 0:
            target_idx = valid_targets[0] # Pick the first one
            plt.figure(figsize=(10, 6))
            shap.waterfall_plot(shap_values[target_idx], max_display=12, show=False)
            plt.title(f"Local Explainability: Gather-Scatter Detection - {tag}")
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"shap_waterfall_local_{tag}.png"), dpi=300, bbox_inches='tight')
            plt.close()
            log(f"  -> Saved Local SHAP plot: {output_dir}/shap_waterfall_local_{tag}.png")


def run_experiment(tag, X_full, names_full, cols, d, burn_end, tr_end, va_end, typ, out_dir, use_gnn):
    os.makedirs(out_dir, exist_ok=True)
    n, y = d["n"], d["y"]
    names = [names_full[i] for i in cols]
    X = X_full[:, cols]                                                      # copy of the chosen columns
    mu, sd = X[burn_end:tr_end].mean(0), X[burn_end:tr_end].std(0) + 1e-6   # TRAIN statistics only
    X -= mu
    X /= sd
    ytr, yva, yte = y[:tr_end], y[tr_end:va_end], y[va_end:]
    Xva, Xte = X[tr_end:va_end], X[va_end:]
    print("\n" + "#" * 80 + f"\n# EXPERIMENT '{tag}': {X.shape[1]} features, GNN {'ON' if use_gnn else 'off'}\n" + "#" * 80)

    log(f"[{tag}] tabular models")
    runs = [fit_xgb("tabular (all train rows)", "tab", X[:tr_end], ytr, Xva, yva, Xte, yte)]
    if burn_end >= int(0.01 * n) and y[burn_end:tr_end].sum() >= 300:
        runs.append(fit_xgb("tabular (burn-in applied)", "tab-burn", X[burn_end:tr_end], y[burn_end:tr_end],
                            Xva, yva, Xte, yte))
    best_tab = max(runs, key=lambda r: r["ap_val"])

    audit = None
    if AUDIT_FINGERPRINT_ONLY:
        fpc = [i for i, nm in enumerate(names) if is_fingerprint(nm)]
        if fpc:
            audit = fit_xgb("AUDIT: fingerprint columns only", "audit", X[:tr_end][:, fpc], ytr, Xva[:, fpc], yva,
                            Xte[:, fpc], yte)

    gnn = None
    if use_gnn and setup_gnn_deps():
        log(f"[{tag}] GNN stage")
        try:
            gnn = run_gnn_stage(X, d, tr_end, va_end, burn_end if GNN_BURN_IN else 0)
        except ImportError as e:
            log(f"GNN stage skipped: {e}")
    blend = None
    if gnn is not None:
        g = gnn["final"]
        runs.append(make_run("GNN alone", "gnn", g[tr_end:va_end], g[va_end:], yva, yte))
        blend = blend_run(best_tab, g[tr_end:va_end], g[va_end:], yva, yte)
        runs.append(blend)
        if gnn["oof"] is not None:
            s0 = gnn["s0"]
            Xs = np.hstack([X[s0:], gnn["oof"][s0:, None]])
            a, b = tr_end - s0, va_end - s0
            Xs[a:, -1] = g[tr_end:]
            runs.append(fit_xgb("stacked OOF (tabular + GNN score)", "stack", Xs[:a], y[s0:tr_end], Xs[a:b], yva,
                                Xs[b:], yte))

    print(f"\n[{tag}] results (headline = best VALIDATION PR-AUC)")
    print("{:<42} {:>8} {:>8} {:>7} {:>7} {:>7}".format("run", "val AP", "test AP", "prec", "recall", "F1"))
    for r in runs + ([audit] if audit else []):
        print("{:<42} {:>8.4f} {:>8.4f} {:>7.3f} {:>7.3f} {:>7.3f}".format(
            r["name"], r["ap_val"], r["ap_test"], r["precision"], r["recall"], r["f1"]))
    head = max(runs, key=lambda r: r["ap_val"])
    blk = max(len(yte) // 100, 1)

    gnn_gain = None
    if blend is not None:
        gnn_run = [r for r in runs if r["short"] == "gnn"][0]
        print(f"\n[{tag}] WHAT DOES THE GNN ADD?  (paired bootstrap on test PR-AUC, {N_BOOT} resamples)")
        for label, a_, b_ in (("blend minus best tabular", blend, best_tab), ("GNN alone minus best tabular", gnn_run, best_tab)):
            m_, (lo_, hi_), pg = paired_bootstrap_ap(yte, a_["p_te"], b_["p_te"], N_BOOT, block=blk)
            print(f"  {label:<32} diff {m_:+.4f}  95% time-block CI [{lo_:+.4f}, {hi_:+.4f}]  P(>0) {pg:.2f}")
        print(f"  blend weights (tabular, GNN): {blend['weights'][0]:+.2f}, {blend['weights'][1]:+.2f}")
        gnn_gain = dict(blend_minus_tab=float(blend["ap_test"] - best_tab["ap_test"]),
                        gnn_alone_minus_tab=float(gnn_run["ap_test"] - best_tab["ap_test"]))

    print(f"\n[{tag}] paired bootstrap: headline minus each other run")
    for r in runs:
        if r is head:
            continue
        for lab, bk in (("iid", 0), ("time-block", blk)):
            m_, (lo_, hi_), pg = paired_bootstrap_ap(yte, head["p_te"], r["p_te"], N_BOOT, block=bk)
            print(f"  vs {r['name']:<40} [{lab:<10}] diff {m_:+.4f}  95% CI [{lo_:+.4f}, {hi_:+.4f}]  P {pg:.2f}")

    diagnostics(head, yva, yte)
    print_importance(best_tab, names)
    if typ is not None:
        typology_table(runs, yte, typ[va_end:])

    summary = {r["name"]: {k: v for k, v in r.items() if k not in ("p_va", "p_te", "model")} for r in runs}
    if audit:
        summary[audit["name"]] = {k: v for k, v in audit.items() if k not in ("p_va", "p_te", "model")}
    summary.update(headline=head["name"], features=names, gnn_gain=gnn_gain)
    with open(os.path.join(out_dir, "results.json"), "w") as fh:
        json.dump(summary, fh, indent=2, default=float)
    best_tab["model"].save_model(os.path.join(out_dir, "xgb_tabular.json"))
    np.savez_compressed(os.path.join(out_dir, "all_runs_test_scores.npz"), y_test=yte,
                        **{r["short"].replace("-", "_"): r["p_te"] for r in runs})

    if gnn is not None and STACK_OOF:
        generate_explainability(runs, X, gnn, names, yte, typ[va_end:] if typ is not None else None, va_end, out_dir, tag)

    return dict(tag=tag, head=head["name"], ap_val=head["ap_val"], ap_test=head["ap_test"], precision=head["precision"],
                recall=head["recall"], f1=head["f1"], gnn_gain=gnn_gain,
                best_tab=(best_tab["name"], best_tab["ap_val"], best_tab["ap_test"]),
                audit=(audit["ap_val"], audit["ap_test"]) if audit else None)


def main():
    global CSV_PATH, PATTERNS_PATH, OUTPUT_DIR, N_BOOT, USE_GNN, SMOKE, STACK_OOF, FEATURE_SETS
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=CSV_PATH)
    ap.add_argument("--patterns", default=PATTERNS_PATH)
    ap.add_argument("--out", default=OUTPUT_DIR)
    ap.add_argument("--sets", default=",".join(FEATURE_SETS), help="comma list of: structural, full")
    ap.add_argument("--no-gnn", action="store_true")
    ap.add_argument("--stack", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--boot", type=int, default=N_BOOT)
    args, _ = ap.parse_known_args()                                 # ignores Jupyter's "-f kernel.json"
    CSV_PATH, PATTERNS_PATH, OUTPUT_DIR, N_BOOT = args.csv, args.patterns, args.out, args.boot
    FEATURE_SETS = [t.strip() for t in args.sets.split(",") if t.strip()]
    USE_GNN, SMOKE, STACK_OOF = USE_GNN and not args.no_gnn, SMOKE or args.smoke, STACK_OOF or args.stack
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    np.random.seed(SEED)
    t_start = time.time()

    if SMOKE:
        CSV_PATH = os.path.join(OUTPUT_DIR, "synthetic_trans.csv")
        PATTERNS_PATH = os.path.join(OUTPUT_DIR, "synthetic_patterns.txt")
        make_synthetic(CSV_PATH, PATTERNS_PATH)
    else:
        if MOUNT_DRIVE:
            from google.colab import drive
            drive.mount("/content/drive")
        CSV_PATH = find_or_upload(CSV_PATH)
        PATTERNS_PATH = find_or_upload(PATTERNS_PATH, required=False)
    log(f"numba: {'available' if HAVE_NUMBA else 'NOT available (distinct features use a slower path)'}")

    log("1/4 Loading + chronological sort")
    d = load_data(CSV_PATH)
    n, y = d["n"], d["y"]
    tr_end, va_end = int(TRAIN_FRAC * n), int((TRAIN_FRAC + VAL_FRAC) * n)
    log(f"  {n:,} edges, {d['N']:,} accounts | train {tr_end:,}  val {va_end - tr_end:,}  test {n - va_end:,}")
    for nm, sl in (("train", slice(0, tr_end)), ("val", slice(tr_end, va_end)), ("test", slice(va_end, n))):
        log(f"  {nm}: illicit {int(y[sl].sum()):,} ({y[sl].mean() * 100:.3f}%)")
    log(f"  time range in this file: {pd.Timestamp(int(d['ts_abs'].min()), unit='s')} -> "
        f"{pd.Timestamp(int(d['ts_abs'].max()), unit='s')}   (full HI-Small: 5,078,345 edges, 5,177 laundering)")

    log("2/4 Causal features (all columns; feature sets are chosen per experiment)")
    X, names, extras = causal_features(d)
    log(f"  {X.shape[1]} features")
    cold = extras["s_new"] | extras["r_new"]
    burn_end, rates = choose_burn_in(cold, tr_end, n)
    log("  cold-start (unseen sender or receiver) per 1% of timeline, train: "
        + " ".join(f"{r:.0%}" for r in rates[:12]) + " ...")
    log(f"  cold-start val {cold[tr_end:va_end].mean():.2%} | test {cold[va_end:].mean():.2%}")
    log(f"  burn-in: first {burn_end:,} rows ({burn_end / max(tr_end, 1):.0%} of train) keep history only; "
        f"positives set aside: {int(y[:burn_end].sum()):,}")
    del extras
    typ = load_typologies(PATTERNS_PATH, d) if PATTERNS_PATH else None      # REPORTING ONLY - never a feature
    del d["df"]

    fp = [is_fingerprint(nm) for nm in names]
    all_cols = list(range(len(names)))
    sets = {"structural": [i for i, f_ in enumerate(fp) if not f_], "full": all_cols}
    log(f"  structural set: {len(sets['structural'])} features | full set: {len(all_cols)} "
        f"(fingerprint columns: {sum(fp)})")
    if not any(fp):
        log("  no IBM fingerprint fields in this file -> 'structural' and 'full' are identical; running once")
        sets = {"structural": all_cols}

    log("3/4 Experiments")
    results = {}
    for tag in FEATURE_SETS:
        if tag in sets:
            results[tag] = run_experiment(tag, X, names, sets[tag], d, burn_end, tr_end, va_end, typ,
                                          os.path.join(OUTPUT_DIR, tag), USE_GNN and tag in GNN_FEATURE_SETS)

    log("4/4 Summary")
    print("\n" + "=" * 96 + "\nSUMMARY (headline per experiment is chosen on VALIDATION PR-AUC)\n" + "=" * 96)
    print("{:<11} {:<40} {:>8} {:>8} {:>6} {:>6} {:>6}".format("features", "headline run", "val AP", "test AP", "prec", "rec", "F1"))
    for tag, r in results.items():
        print("{:<11} {:<40} {:>8.4f} {:>8.4f} {:>6.3f} {:>6.3f} {:>6.3f}".format(
            tag, r["head"][:40], r["ap_val"], r["ap_test"], r["precision"], r["recall"], r["f1"]))
    for tag, r in results.items():
        if r["audit"]:
            print(f"  [{tag}] AUDIT - fingerprint columns ALONE reach val AP {r['audit'][0]:.4f} / test AP {r['audit'][1]:.4f}")
        if r["gnn_gain"]:
            g_ = r["gnn_gain"]
            print(f"  [{tag}] GNN contribution on test AP: blend {g_['blend_minus_tab']:+.4f} vs best tabular, "
                  f"GNN alone {g_['gnn_alone_minus_tab']:+.4f}")
    with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as fh:
        json.dump(results, fh, indent=2, default=float)
    log(f"Done in {(time.time() - t_start) / 60:.1f} min. Outputs in {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Mounted at /content/drive
[04:25:43] numba: available
[04:25:43] 1/4 Loading + chronological sort
[04:26:04]   raw IBM file detected: payment format, currencies, banks and amounts are used
[04:26:11]   5,078,345 edges, 515,088 accounts | train 3,047,007  val 1,015,669  test 1,015,669
[04:26:11]   train: illicit 2,298 (0.075%)
[04:26:11]   val: illicit 1,082 (0.107%)
[04:26:11]   test: illicit 1,797 (0.177%)
[04:26:11]   time range in this file: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00   (full HI-Small: 5,078,345 edges, 5,177 laundering)
[04:26:11] 2/4 Causal features (all columns; feature sets are chosen per experiment)
[04:28:26]   106 features
[04:28:26]   cold-start (unseen sender or receiver) per 1% of timeline, train: 95% 85% 77% 69% 64% 58% 47% 36% 32% 29% 27% 24% ...
[04:28:26]   cold-start val 0.12% | test 0.12%
[04:28:26]   burn-in: first 1,066,452 rows (35% of train) keep history only; positives set aside: 311
[04:28:27]   typology match: 3,209 of 5,177 labelled laundering 

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [04:28:57] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[04:28:59] XGBoost [tabular (burn-in applied)]  train=1,980,555 (pos 1,987)  device=cuda
[0]	validation_0-aucpr:0.32794
[200]	validation_0-aucpr:0.55134
[400]	validation_0-aucpr:0.55735
[600]	validation_0-aucpr:0.56283
[800]	validation_0-aucpr:0.56314
[810]	validation_0-aucpr:0.56328
[04:29:15] Installing torch_geometric + pyg-lib for torch 2.11.0+cu130 ...
[04:29:31] [structural] GNN stage
[04:29:31] GNN stage on cuda
[04:29:32]   [fold1] fit rows 1,523,503 | positives 524
[04:29:34]   sanity checks passed (seed alignment, temporal sampling)
[04:29:35]   [fold1] epoch 01/50  loss 0.5313
[04:29:41]   [fold1] epoch 05/50  loss 0.2965
[04:29:49]   [fold1] epoch 10/50  loss 0.2373
[04:29:57]   [fold1] epoch 15/50  loss 0.2080
[04:30:05]   [fold1] epoch 20/50  loss 0.1916
[04:30:13]   [fold1] epoch 25/50  loss 0.1747
[04:30:21]   [fold1] epoch 30/50  loss 0.1693
[04:30:29]   [fold1] epoch 35/50  loss 0.1633
[04:30:37]   [fold1] epoch 40/50  loss 0.1490
[04:30:45]   [fold1] epoch 45/50  los

/tmp/ipykernel_5815/1895557075.py:856: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_test_stacked, feature_names=feature_names, show=False)


[05:04:38]   -> Saved Global SHAP plot: /content/aml_outputs/structural/shap_summary_global_structural.png
[05:04:39]   -> Saved Local SHAP plot: /content/aml_outputs/structural/shap_waterfall_local_structural.png

################################################################################
# EXPERIMENT 'full': 106 features, GNN off
################################################################################
[05:04:45] [full] tabular models
[05:04:45] XGBoost [tabular (all train rows)]  train=3,047,007 (pos 2,298)  device=cuda
[0]	validation_0-aucpr:0.32468
[200]	validation_0-aucpr:0.57612
[400]	validation_0-aucpr:0.57884
[404]	validation_0-aucpr:0.57860
[05:05:02] XGBoost [tabular (burn-in applied)]  train=1,980,555 (pos 1,987)  device=cuda
[0]	validation_0-aucpr:0.27728
[200]	validation_0-aucpr:0.58254
[358]	validation_0-aucpr:0.58400
[05:05:14] XGBoost [AUDIT: fingerprint columns only]  train=3,047,007 (pos 2,298)  device=cuda
[0]	validation_0-aucpr:0.01057
[103]	validation_